# Power Grid Stability — Inertial Kuramoto

The swing equation models power grid transient stability:

$$m_i \ddot{\theta}_i + d_i \dot{\theta}_i = P_i + \sum_j K_{ij} \sin(\theta_j - \theta_i)$$

This notebook demonstrates:
1. Balanced grid stays synchronized
2. Generator trip causes frequency deviation
3. Weak transmission lines cause desynchronization

In [ ]:
import numpy as np
from scpn_phase_orchestrator.upde.inertial import InertialKuramotoEngine

In [ ]:
# 4-bus power grid: 2 generators, 2 loads
N = 4
engine = InertialKuramotoEngine(N, dt=0.01)

theta = np.zeros(N)
omega_dot = np.zeros(N)
power = np.array([1.0, 1.0, -1.0, -1.0])  # gen, gen, load, load
knm = np.ones((N, N)) * 2.0
np.fill_diagonal(knm, 0.0)
inertia = np.ones(N) * 5.0
damping = np.ones(N) * 1.0

## Scenario 1: Balanced Grid

In [ ]:
ft, fo, theta_traj, omega_traj = engine.run(
    theta, omega_dot, power, knm, inertia, damping, 500
)
print(f"Coherence R: {engine.coherence(ft):.3f}")
print(f"Freq deviation: {engine.frequency_deviation(fo):.4f} Hz")

## Scenario 2: Generator Trip

Generator 0 suddenly loses all power output.

In [ ]:
power_trip = power.copy()
power_trip[0] = 0.0  # generator 0 trips

ft2, fo2, _, omega_traj2 = engine.run(
    theta, omega_dot, power_trip, knm, inertia, damping, 500
)
print(f"Coherence R: {engine.coherence(ft2):.3f}")
print(f"Freq deviation: {engine.frequency_deviation(fo2):.4f} Hz")
print(f"Max transient: {np.max(np.abs(omega_traj2)) / (2 * np.pi):.4f} Hz")

## Scenario 3: Weak Transmission Lines

Reduce coupling by 100x — simulates long-distance or degraded lines.

In [ ]:
theta_perturbed = np.array([0.0, 0.5, 1.0, 1.5])
knm_weak = knm * 0.01

ft3, fo3, _, _ = engine.run(
    theta_perturbed, omega_dot, power, knm_weak, inertia, damping, 500
)
R = engine.coherence(ft3)
print(f"Coherence R: {R:.3f}")
print(f"Freq deviation: {engine.frequency_deviation(fo3):.4f} Hz")
if R < 0.5:
    print("WARNING: Grid desynchronized — cascading failure risk")